# Public Data Context

This notebook assembles the public data used to frame a hypothetical 100 MW data center in Loudoun County, Virginia. I use EPA eGRID data to establish the regional electricity emissions factor, Census data to describe broad county demographics, and FEMA data for an initial hazard screen. These inputs provide planning context, not a parcel-level site assessment.

## Regional electricity emissions

Loudoun County is located in the SRVC eGRID subregion. I extract the 2023 total-output CO₂ rate from the EPA workbook and save it as the emissions factor used in the scenario model.

In [1]:
import pandas as pd
from pathlib import Path
from datetime import date

# Define project folders and files

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
TABLES = ROOT / "outputs" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

input_path = RAW / "egrid2023_data_rev2.xlsx"
output_path = PROCESSED / "egrid_emissions_factor.csv"

# Confirm that the Excel file exists (simple validation step)

if not input_path.exists():
    raise FileNotFoundError(
        f"Could not find the eGRID workbook at:\n{input_path}\n\n"
        "Confirm that egrid2023_data_rev2.xlsx is saved "
        "inside the data/raw folder."
    )

# Read the eGRID subregion worksheet

egrid = pd.read_excel(
    input_path,
    sheet_name="SRL23",
    header=1,
    engine="openpyxl"
)

print(f"Loaded {len(egrid):,} eGRID subregion records.")
print(f"Source file: {input_path.name}")

# Standardize the column names

egrid.columns = (
    egrid.columns
    .astype(str)
    .str.strip()
    .str.upper()
)

# Confirm that the required columns are present

required_columns = {
    "YEAR",
    "SUBRGN",
    "SRNAME",
    "SRCO2RTA"
}

missing_columns = required_columns - set(egrid.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}\n\n"
        f"Available columns are:\n{egrid.columns.tolist()}"
    )

# Clean the subregion field

egrid["SUBRGN"] = (
    egrid["SUBRGN"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Select the SRVC subregion

srvc = egrid.loc[egrid["SUBRGN"] == "SRVC"].copy()

if len(srvc) != 1:
    raise ValueError(
        f"Expected exactly one SRVC row, but found {len(srvc)}."
    )

print("\nSelected eGRID record:")

display(
    srvc[
        [
            "YEAR",
            "SUBRGN",
            "SRNAME",
            "SRCO2RTA"
        ]
    ]
)

# Extract the CO2 output emission factor

raw_factor = srvc.iloc[0]["SRCO2RTA"]

co2_factor = pd.to_numeric(
    str(raw_factor).replace(",", "").strip(),
    errors="raise"
)

source_year = int(srvc.iloc[0]["YEAR"])

# Run quality checks

assert srvc.iloc[0]["SUBRGN"] == "SRVC"
assert pd.notna(co2_factor)
assert 0 < co2_factor < 3000
assert source_year == 2023

print(
    f"\nSRVC CO2 total output emission rate: "
    f"{co2_factor:,.3f} lb CO2/MWh"
)

# Create the standardized output table

egrid_factor = pd.DataFrame(
    [
        {
            "project_zip": "20166",
            "project_location": (
                "Sterling/Dulles area, "
                "Loudoun County, Virginia"
            ),
            "source_year": source_year,
            "egrid_subregion": srvc.iloc[0]["SUBRGN"],
            "egrid_subregion_name": srvc.iloc[0]["SRNAME"],
            "co2_lb_per_mwh": co2_factor,
            "source_file": input_path.name,
            "source_worksheet": "SRL23",
            "source_field": "SRCO2RTA",
            "source_url": (
                "https://www.epa.gov/egrid/detailed-data"
            ),
            "access_date": date.today().isoformat(),
            "data_status": "EPA eGRID public data"
        }
    ]
)

print("\nStandardized output:")

display(egrid_factor)

# Save the processed emission factor

egrid_factor.to_csv(
    output_path,
    index=False
)

Loaded 27 eGRID subregion records.
Source file: egrid2023_data_rev2.xlsx

Selected eGRID record:


,YEAR,SUBRGN,SRNAME,SRCO2RTA
26,2023,SRVC,SERC Virginia/Carolina,593.419



SRVC CO2 total output emission rate: 593.419 lb CO2/MWh

Standardized output:


,project_zip,project_location,source_year,egrid_subregion,egrid_subregion_name,co2_lb_per_mwh,source_file,source_worksheet,source_field,source_url,access_date,data_status
0,20166,"Sterling/Dulles area, Loudoun County, Virginia",2023,SRVC,SERC Virginia/Carolina,593.419,egrid2023_data_rev2.xlsx,SRL23,SRCO2RTA,https://www.epa.gov/egrid/detailed-data,2026-10-13,EPA eGRID public data


## County demographic context

I use 2024 ACS five-year estimates for population, median household income, and poverty rate. These county-level measures provide a starting point for thinking about community context and engagement, but they do not describe conditions around a specific proposed site.

In [2]:
population = pd.read_csv(
    RAW / "acs_2024_loudoun_population.csv"
)

income = pd.read_csv(
    RAW / "acs_2024_loudoun_income.csv"
)

poverty = pd.read_csv(
    RAW / "acs_2024_loudoun_poverty.csv"
)

print("Population columns:")
print(population.columns.tolist())

print("\nIncome columns:")
print(income.columns.tolist())

print("\nPoverty columns:")
print(poverty.columns.tolist())

Population columns:
['GEO_ID', 'NAME', 'B01003_001E', 'B01003_001M', 'Unnamed: 4']

Income columns:
['GEO_ID', 'NAME', 'B19013_001E', 'B19013_001M', 'Unnamed: 4']

Poverty columns:
['GEO_ID', 'NAME', 'B17001_001E', 'B17001_001M', 'B17001_002E', 'B17001_002M', 'B17001_003E', 'B17001_003M', 'B17001_004E', 'B17001_004M', 'B17001_005E', 'B17001_005M', 'B17001_006E', 'B17001_006M', 'B17001_007E', 'B17001_007M', 'B17001_008E', 'B17001_008M', 'B17001_009E', 'B17001_009M', 'B17001_010E', 'B17001_010M', 'B17001_011E', 'B17001_011M', 'B17001_012E', 'B17001_012M', 'B17001_013E', 'B17001_013M', 'B17001_014E', 'B17001_014M', 'B17001_015E', 'B17001_015M', 'B17001_016E', 'B17001_016M', 'B17001_017E', 'B17001_017M', 'B17001_018E', 'B17001_018M', 'B17001_019E', 'B17001_019M', 'B17001_020E', 'B17001_020M', 'B17001_021E', 'B17001_021M', 'B17001_022E', 'B17001_022M', 'B17001_023E', 'B17001_023M', 'B17001_024E', 'B17001_024M', 'B17001_025E', 'B17001_025M', 'B17001_026E', 'B17001_026M', 'B17001_027E', 'B170

In [3]:
population = pd.read_csv(
    RAW / "acs_2024_loudoun_population.csv",
    dtype=str
)

income = pd.read_csv(
    RAW / "acs_2024_loudoun_income.csv",
    dtype=str
)

poverty = pd.read_csv(
    RAW / "acs_2024_loudoun_poverty.csv",
    dtype=str
)

population_row = population.loc[
    population["NAME"]
    .str.contains("Loudoun County", case=False, na=False)
].copy()

income_row = income.loc[
    income["NAME"]
    .str.contains("Loudoun County", case=False, na=False)
].copy()

poverty_row = poverty.loc[
    poverty["NAME"]
    .str.contains("Loudoun County", case=False, na=False)
].copy()

# Verify that each file contains exactly one Loudoun row 

assert len(population_row) == 1
assert len(income_row) == 1
assert len(poverty_row) == 1

# Define a helper for converting Census values

def census_number(dataframe, column):
    value = dataframe.iloc[0][column]

    return pd.to_numeric(
        str(value).replace(",", "").strip(),
        errors="raise"
    )

# Extract the required estimates

total_population = census_number(
    population_row,
    "B01003_001E"
)

median_household_income = census_number(
    income_row,
    "B19013_001E"
)

poverty_universe = census_number(
    poverty_row,
    "B17001_001E"
)

population_below_poverty = census_number(
    poverty_row,
    "B17001_002E"
)

# Calculate the poverty percentage

poverty_rate_pct = (
    100
    * population_below_poverty
    / poverty_universe
)

# Create one clean county-context table

acs_context = pd.DataFrame(
    [
        {
            "geography": "Loudoun County, Virginia",
            "source_year": 2024,
            "population": int(total_population),
            "median_household_income_usd": int(
                median_household_income
            ),
            "poverty_status_universe": int(
                poverty_universe
            ),
            "population_below_poverty": int(
                population_below_poverty
            ),
            "poverty_rate_pct": round(
                poverty_rate_pct,
                2
            ),
            "data_status": (
                "2024 ACS five-year estimates"
            )
        }
    ]
)

# Display the result

display(acs_context)

print(
    f"Population: "
    f"{total_population:,.0f}"
)

print(
    f"Median household income: "
    f"${median_household_income:,.0f}"
)

print(
    f"Population below poverty: "
    f"{population_below_poverty:,.0f}"
)

print(
    f"Poverty rate: "
    f"{poverty_rate_pct:.2f}%"
)

output_path = (
    PROCESSED
    / "loudoun_acs_context.csv"
)

acs_context.to_csv(
    output_path,
    index=False
)

,geography,source_year,population,median_household_income_usd,poverty_status_universe,population_below_poverty,poverty_rate_pct,data_status
0,"Loudoun County, Virginia",2024,432998,181765,430965,18249,4.23,2024 ACS five-year estimates


Population: 432,998
Median household income: $181,765
Population below poverty: 18,249
Poverty rate: 4.23%


## County-level hazard screen

The FEMA National Risk Index provides a first-pass view of Loudoun County’s relative natural-hazard exposure. The available file includes an overall risk score and a heat-wave risk score. I use these as screening indicators rather than estimates of risk to a particular parcel or facility.

In [4]:
nri_raw = pd.read_csv(
    RAW / "fema_nri_county.csv",
    low_memory=False
)

print([
    col for col in nri_raw.columns
    if col.upper().startswith(("HWAV", "RFLD", "CFLD"))
    and ("RISK" in col.upper())
])

['CFLD_RISKV', 'CFLD_RISKS', 'CFLD_RISKR', 'HWAV_RISKV', 'HWAV_RISKS', 'HWAV_RISKR']


In [5]:
loudoun_fema = nri_raw[
    nri_raw["STATE"].astype(str).str.contains(
        "VA|Virginia",
        case=False,
        regex=True
    )
    & nri_raw["COUNTY"].astype(str).str.contains(
        "Loudoun",
        case=False,
        regex=False
    )
].copy()

columns_to_keep = [
    "STATE",
    "COUNTY",
    "RISK_SCORE",
    "RISK_RATNG",
    "HWAV_RISKS",
    "HWAV_RISKR",
    "RFLD_RISKS",
    "RFLD_RISKR"
]

columns_to_keep = [
    col for col in columns_to_keep
    if col in loudoun_fema.columns
]

loudoun_fema = loudoun_fema[columns_to_keep]

loudoun_fema.to_csv(
    PROCESSED / "loudoun_fema_risk.csv",
    index=False
)

display(loudoun_fema)

,STATE,COUNTY,RISK_SCORE,RISK_RATNG,HWAV_RISKS,HWAV_RISKR
2865,Virginia,Loudoun,78.117048,Relatively Low,91.104689,Relatively Moderate


## Combined screening table

The following table brings the public indicators and the facility-size assumption into one place. The `data_type` field distinguishes observed public data from values created specifically for this hypothetical analysis.

In [6]:
acs = pd.read_csv(
    PROCESSED / "loudoun_acs_context.csv"
)

fema = pd.read_csv(
    PROCESSED / "loudoun_fema_risk.csv"
)

egrid = pd.read_csv(
    PROCESSED / "egrid_emissions_factor.csv"
)

# Confirm that each file contains one row

assert len(acs) == 1
assert len(fema) == 1
assert len(egrid) == 1

# Extract the values used in the context table

population = int(
    acs.loc[0, "population"]
)

median_income = int(
    acs.loc[0, "median_household_income_usd"]
)

poverty_rate = float(
    acs.loc[0, "poverty_rate_pct"]
)

fema_risk_score = float(
    fema.loc[0, "RISK_SCORE"]
)

fema_risk_rating = str(
    fema.loc[0, "RISK_RATNG"]
)

heat_risk_score = float(
    fema.loc[0, "HWAV_RISKS"]
)

heat_risk_rating = str(
    fema.loc[0, "HWAV_RISKR"]
)

co2_rate = float(
    egrid.loc[0, "co2_lb_per_mwh"]
)

egrid_region = str(
    egrid.loc[0, "egrid_subregion"]
)

# The hypothetical facility design assumes a 100 MW maximum demand.
facility_capacity_mw = 100

# Build the context table

context_table = pd.DataFrame(
    [
        {
            "metric": "County population",
            "value": f"{population:,.0f}",
            "unit": "people",
            "source": "U.S. Census Bureau ACS 5-year estimates",
            "year": 2024,
            "data_type": "Public"
        },
        {
            "metric": "Median household income",
            "value": f"{median_income:,.0f}",
            "unit": "2024 USD per household",
            "source": "U.S. Census Bureau ACS 5-year estimates",
            "year": 2024,
            "data_type": "Public"
        },
        {
            "metric": "Population below poverty rate",
            "value": f"{poverty_rate:.1f}",
            "unit": "percent",
            "source": "U.S. Census Bureau ACS 5-year estimates",
            "year": 2024,
            "data_type": "Public"
        },
        {
            "metric": "Overall natural-hazard risk",
            "value": (
                f"{fema_risk_score:.1f} "
                f"({fema_risk_rating})"
            ),
            "unit": "FEMA relative score and rating",
            "source": "FEMA National Risk Index v1.20",
            "year": 2025,
            "data_type": "Public"
        },
        {
            "metric": "Regional electricity CO2 output rate",
            "value": f"{co2_rate:.3f}",
            "unit": "lb CO2 per MWh",
            "source": f"EPA eGRID, subregion {egrid_region}",
            "year": 2023,
            "data_type": "Public"
        },
        {
            "metric": "Maximum facility demand",
            "value": f"{facility_capacity_mw:,.0f}",
            "unit": "MW",
            "source": "Project design assumption",
            "year": "N/A",
            "data_type": "Illustrative"
        },
        {
            "metric": "Heat-wave risk",
            "value": (
                f"{heat_risk_score:.1f} "
                f"({heat_risk_rating})"
            ),
            "unit": "FEMA relative score and rating",
            "source": "FEMA National Risk Index v1.20",
            "year": 2025,
            "data_type": "Public"
        },
    ]
)

display(context_table)

output_path = TABLES / "public_context_table.csv"

context_table.to_csv(
    output_path,
    index=False
)

,metric,value,unit,source,year,data_type
0,County population,"432,998",people,U.S. Census Bureau ACS 5-year estimates,2024,Public
1,Median household income,"181,765",2024 USD per household,U.S. Census Bureau ACS 5-year estimates,2024,Public
2,Population below poverty rate,4.2,percent,U.S. Census Bureau ACS 5-year estimates,2024,Public
3,Overall natural-hazard risk,78.1 (Relatively Low),FEMA relative score and rating,FEMA National Risk Index v1.20,2025,Public
4,Regional electricity CO2 output rate,593.419,lb CO2 per MWh,"EPA eGRID, subregion SRVC",2023,Public
5,Maximum facility demand,100,MW,Project design assumption,N/A,Illustrative
6,Heat-wave risk,91.1 (Relatively Moderate),FEMA relative score and rating,FEMA National Risk Index v1.20,2025,Public


## Interpretation and limits

Loudoun County has approximately 433,000 residents, a median household income of about $182,000, and a countywide poverty rate of 4.2%. FEMA classifies its overall natural-hazard risk as relatively low, although the available heat-wave indicator is relatively moderate. The SRVC electricity region has a 2023 output rate of 593.419 pounds of CO₂ per MWh, which establishes the location-based emissions factor used later in the analysis.

These indicators establish the scale and context of the hypothetical decision. The demographic measures identify questions that may deserve attention during community engagement, while the FEMA data identify hazards that would require more detailed site and engineering review. I use 100 MW as a round-number design capacity for the hypothetical facility; it is not an estimate for a specific proposed project.

The data cannot determine whether a specific parcel is suitable, whether the utility can serve the proposed load, or what tariff would apply. County averages can also obscure neighborhood-level conditions, and demographic indicators do not measure community attitudes or support. A real project would require parcel-level hazard data, utility coordination, engineering analysis, and direct community engagement.